# Generate the global oracle comparison

A mask selecting exactly $R$ of $N$ channels has $\binom NR$ possibilities. We derive a dynamic program for $I-qL$, recover a mask, and use ratio bounds to find the best importance/latency ratio. Then we compare the oracle with Paper greedy, saturation tiles and Top-$R$.

Run from this notebook's directory, after [02_capture_traces.ipynb](02_capture_traces.ipynb). The inputs are `activation_traces.npz` and the cost model from [Notebook 01](../01/01_chunk_latency.ipynb). The final cell saves `oracle_results_fixed_r.csv`; [02_analyze_oracle.ipynb](02_analyze_oracle.ipynb) reads it to draw the figures.

Latency throughout is the additive model estimate.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from scipy.ndimage import maximum_filter1d

MODEL = Path("../01/latency_model.json")
TRACE = Path("activation_traces.npz")
RESULTS = Path("oracle_results_fixed_r.csv")
UPSTREAM = Path("../../../upstream/vlm-flash")
PAPER_IMPL = "native"
MAX_TRACES_PER_SHAPE = 32  # 0: use every archived trace.
BUDGETS = np.arange(1, 10) / 10

## Read activation inputs

Read the vectors from [02_capture_traces.ipynb](02_capture_traces.ipynb). Use up to `MAX_TRACES_PER_SHAPE` evenly spaced traces per shape, or all traces when it is zero. Each vector keeps all channels and is normalized to mean one.

For a projection with $N$ inputs and $d$ outputs, the row length $w$ is $d$ times the bytes per weight, divided by 1024. Layers and budgets from the same prompt are related observations.

In [ ]:
with np.load(TRACE, allow_pickle=False) as archive:
    manifest = json.loads(str(archive["manifest_json"].item()))
    traces = [{**t, "values": archive[t["array"]].astype(float)} for t in manifest["traces"]]
if MAX_TRACES_PER_SHAPE:
    groups = [[t for t in traces if t["shape"] == shape] for shape in sorted({t["shape"] for t in traces})]
    traces = [g[i] for g in groups for i in np.linspace(
        0, len(g) - 1, min(len(g), MAX_TRACES_PER_SHAPE), dtype=int)]
traces.sort(key=lambda t: t["trace_id"])
weight_bytes = {"float16": 2, "bfloat16": 2, "float32": 4}[manifest["model_dtype"]]

## Objective and cost model

Let $v_i\ge0$ be the importance of channel $i$, and let $\mathcal C(M)$ contain the maximal contiguous runs selected by a binary mask $M$. Define

$$I(M)=\sum_{i=0}^{N-1}v_iM_i,\qquad L(M)=\sum_{C\in\mathcal C(M)}T(|C|),\qquad
\eta_R^\star=\max_{\|M\|_1=R}\frac{I(M)}{L(M)}.$$

For a requested fraction $f$, set $R=\max(1,\lfloor fN\rfloor)\le N$. Adjacent selected intervals form one run and are charged once. The oracle assumes the continuous two-line cost

$$T(0)=0,\qquad T(\ell)=\begin{cases}a+b_1\ell,&1\le\ell\le s,\\b_2\ell,&\ell>s,\end{cases}
\qquad a=(b_2-b_1)s,\quad 0\le b_1\le b_2,\quad b_2>0.$$

The input model is $T_{\rm KiB}(z)=c_1z+\delta\max(z_{\rm 2L},z)$ in milliseconds. For a weight row of $w$ KiB, substituting $z=w\ell$ gives

$$s=\frac{z_{\rm 2L}}w,\qquad b_1=wc_1,\qquad b_2=w(c_1+\delta).$$

The boundary $s$ remains real-valued in the cost. Lengths outside the measured range use the model's linear extension. The coefficients below are the fixed input to this experiment.

In [ ]:
model = json.loads(MODEL.read_text())
s_kib = model["s_kib"]
c1 = model["c1_ms_per_kib"]
delta = model["delta_ms_per_kib"]
s99 = model["s99_kib"]

In [ ]:
def costs(n, s, b1, b2):
    length = np.arange(n + 1)
    T = b2 * length + (b2 - b1) * np.maximum(s - length, 0)
    T[0] = 0
    return T


def runs(mask):
    edges = np.flatnonzero(np.diff(np.r_[False, mask, False]))
    return edges.reshape(-1, 2)


def metrics(v, mask, T):
    spans = runs(mask)
    return v[mask].sum(), T[spans[:, 1] - spans[:, 0]].sum()

## An additive subproblem

For a proposed ratio $q\ge0$, define

$$F_R(q)=\max_{\|M\|_1=R}\{I(M)-qL(M)\}.$$

Every feasible latency is positive, so $F_R(q)\ge0$ exactly when $q\le\eta_R^\star$. One additive optimization therefore tests whether a ratio is attainable.

Write $Z=N-R$ and $P_j=\sum_{i<j}v_i$. Let $D_{z,r}$ be the largest $I-qL$ over prefixes of length $z+r$ with $z$ zeros and $r$ ones. With no zeros,

$$D_{0,r}=P_r-qT(r).$$

For $z>0$, place the last zero after a prefix containing $z-1$ zeros and $t$ ones. The final run then occupies $[z+t,z+r)$ and has length $r-t$. Allowing an empty final run gives

$$\boxed{D_{z,r}=\max_{0\le t\le r}\left[D_{z-1,t}+P_{z+r}-P_{z+t}-qT(r-t)\right].}$$

Every feasible prefix has this decomposition. For a fixed $t$, replacing its earlier prefix by the best $D_{z-1,t}$ leaves the final run and its cost unchanged. Conversely, an optimal earlier prefix, one zero and the final run form a feasible prefix. Induction gives $F_R(q)=D_{Z,R}$; the separating zero keeps adjacent pieces from being charged separately.

There are $(Z+1)(R+1)$ states. Scanning all $t$ takes $O((Z+1)(R+1)^2)$ time for a general run-length table. The two-line form reduces this cost.

## Two range maxima per row

For a fixed $z$, define

$$A_t=D_{z-1,t}-P_{z+t}+qb_1t,\qquad B_t=D_{z-1,t}-P_{z+t}+qb_2t.$$

Let $h=\lfloor s\rfloor$; only the integer window is rounded, while $a=(b_2-b_1)s$ stays unchanged. Separating empty, short and long final runs yields

$$D_{z,r}=\max\left\{\begin{aligned}&D_{z-1,r},\\&P_{z+r}-q(a+b_1r)+\max_{\max(0,r-h)\le t<r} A_t,\\&P_{z+r}-qb_2r+\max_{0\le t<r-h} B_t.\end{aligned}\right.$$

An empty maximum is $-\infty$. The short range is a moving window; the long range is a growing prefix. Each row takes $O(R+1)$ time: in a moving-maximum queue, each index enters and leaves at most once.

`trailing_max(values, width)[j]` returns the maximum over indices $\max(0,j-\mathrm{width}+1)$ through $j$. SciPy's linear-time filter provides this window, with `origin` placing it behind the current index. `np.maximum.accumulate` provides the prefix maximum.

In [ ]:
def trailing_max(values, width):
    return maximum_filter1d(values, size=width, origin=(width - 1) // 2,
                            mode="constant", cval=-np.inf)

In [ ]:
def bellman(v, R, q, s, b1, b2):
    Z = len(v) - R
    P = np.r_[0., np.cumsum(v)]
    r = np.arange(R + 1)
    D = np.empty((Z + 1, R + 1))
    D[0] = P[:R + 1] - q * costs(R, s, b1, b2)
    width = min(int(np.floor(s)), R)
    for z in range(1, Z + 1):
        prefix = P[z:z + R + 1]
        previous = D[z - 1] - prefix
        short = np.full(R + 1, -np.inf)
        if width:
            short[1:] = trailing_max(previous[:-1] + q * b1 * r[:-1], width)
        long = np.full(R + 1, -np.inf)
        long[width + 1:] = np.maximum.accumulate(previous + q * b2 * r)[:R - width]
        D[z] = np.maximum.reduce([
            D[z - 1],
            prefix - q * ((b2 - b1) * s + b1 * r) + short,
            prefix - q * b2 * r + long])
    return D

### Recovering a mask

At $(z,r)$, choose a maximizing $t$ in the original recurrence, select $[z+t,z+r)$ and continue at $(z-1,t)$. At $z=0$, the remaining prefix is all ones. Ties may produce different optimal masks.

Keeping the value table and rescanning one row per recovery step gives

$$O((N-R+1)(R+1))\quad\text{time and memory per additive solve, including recovery}.$$

For $R\simeq N/2$ this is quadratic; the bound also covers $R=N$.

In [ ]:
def recover(v, R, q, T, D):
    P = np.r_[0., np.cumsum(v)]
    mask = np.zeros(len(v), bool)
    r = R
    for z in range(len(v) - R, 0, -1):
        t = np.arange(r + 1)
        scores = D[z - 1, :r + 1] + P[z + r] - P[z + t] - q * T[r - t]
        start = int(np.argmax(scores))
        mask[z + start:z + r] = True
        r = start
    mask[:r] = True
    return mask

## Bounding and locating the optimal ratio

Let $M_C$ be the most important interval of exactly $R$ rows, found by prefix sums. For any mask with $R$ selected rows,

$$L(M)=b_1R+(b_2-b_1)\sum_C\max(s,|C|)
\ge b_1R+(b_2-b_1)\max(s,R)=T(R).$$

The contiguous seed attains this minimum cost. Top-$R$ selects the largest individual values and gives the maximum importance. Hence

$$\ell_0=\frac{I(M_C)}{T(R)}\le\eta_R^\star\le u_0=\frac{I(\mathrm{Top}\text{-}R)}{T(R)}.$$

The lower bound belongs to a feasible mask. The upper bound combines the best numerator and denominator, which may come from different masks.

Test $q=(\ell+u)/2$. If $F_R(q)<0$, set $u=q$. Otherwise recover a mask and set $\ell=I(M)/L(M)$. Each step at least halves the bracket. Stopping at $u-\ell\le\epsilon\ell_0$ gives

$$0\le1-\frac{I(M)/L(M)}{\eta_R^\star}\le\epsilon.$$

For nonzero importance, $\ell_0>0$. With $\kappa_0=u_0/\ell_0$, at most $\lceil\log_2(\kappa_0/\epsilon)\rceil$ additive solves suffice. Zero importance returns immediately. After initialization, the search costs $O((N-R+1)(R+1)\log(\kappa_0/\epsilon))$ time. The recurrence is exact mathematically; the implementation evaluates its bounds in float64.

In [ ]:
def top_r(v, R):
    mask = np.zeros(len(v), bool)
    mask[np.argsort(-v, kind="stable")[:R]] = True
    return mask

In [ ]:
def solve(v, R, s, b1, b2, rtol=1e-6):
    T = costs(len(v), s, b1, b2)
    P = np.r_[0., np.cumsum(v)]
    start = np.argmax(P[R:] - P[:-R])
    mask = np.zeros(len(v), bool)
    mask[start:start + R] = True
    I, L = metrics(v, mask, T)
    lower = I / L
    upper = v[top_r(v, R)].sum() / T[R]
    tolerance = rtol * lower
    calls = 0
    while upper - lower > tolerance:
        q = (lower + upper) / 2
        D = bellman(v, R, q, s, b1, b2)
        calls += 1
        if D[-1, -1] >= 0:
            mask = recover(v, R, q, T, D)
            I, L = metrics(v, mask, T)
            lower = I / L
        else:
            upper = q
    return mask, lower, upper, calls

## Comparison selectors

**Paper greedy** ranks candidate windows by importance divided by their modeled cost, then accepts non-overlapping whole windows that fit the remaining budget. The native implementation supplies this selection. `PAPER_GEOMETRY` specifies window start/step and stride cap in KiB for each input-by-output shape. The ceiling is $z_{99}$, the first size reaching 99% of peak smoothed training throughput in the input profile.

Whole windows can leave the budget unfilled: $|M_P|\le R$. We keep that mask as returned. Final costs for every method use maximal merged runs through `metrics`.

In [ ]:
PAPER_GEOMETRY = {"896x128": (8, 8), "896x896": (8, 8),
                  "896x4864": (8, 8), "4864x896": (12, 16)}


def load_paper():
    import sys, torch
    sys.path.insert(0, str((UPSTREAM / "src").resolve()))
    import vlmflash
    table = vlmflash.LatencyTable({
        z: c1*z + delta*max(s_kib, z)
        for z in range(1, int(np.ceil(max(s_kib, s99))) + 2)})

    def select(v, R, w, shape):
        start, jump = PAPER_GEOMETRY[shape]
        params = vlmflash.ChunkParams(start_kb=start, step_kb=start,
                                     jump_cap_kb=jump, end_kb=s99 + 1)
        result = vlmflash.select_chunks(torch.tensor(v, dtype=torch.float32),
            R, w, table, params, impl=PAPER_IMPL)
        return result.mask.cpu().numpy()

    return select

**Saturation tiles** use width $B=\lceil s\rceil$. For each offset $0$ and $\lfloor B/2\rfloor$, choose the $\lfloor R/B\rfloor$ most important full tiles, then the best disjoint interval of length $R\bmod B$. Keep the mask with the larger merged-run ratio. For $R<B$, use the best length-$R$ interval.

Top-$R$, tiles and the oracle select exactly $R$ rows. Paper may select fewer; its output does not change the oracle's budget.

In [ ]:
def tiles(v, R, T, B):
    P = np.r_[0., np.cumsum(v)]
    if R < B:
        start = np.argmax(P[R:] - P[:-R])
        mask = np.zeros(len(v), bool)
        mask[start:start + R] = True
        return mask
    choices = []
    k, remaining = divmod(R, B)
    for offset in (0, B // 2):
        starts = np.arange(offset, len(v) - B + 1, B)
        if len(starts) < k:
            continue
        weights = P[starts + B] - P[starts]
        mask = np.zeros(len(v), bool)
        for start in starts[np.argsort(-weights, kind="stable")[:k]]:
            mask[start:start + B] = True
        if remaining:
            occupied = np.r_[0, np.cumsum(mask)]
            valid = occupied[remaining:] == occupied[:-remaining]
            if not valid.any():
                continue
            scores = np.where(valid, P[remaining:] - P[:-remaining], -np.inf)
            start = np.argmax(scores)
            mask[start:start + remaining] = True
        I, L = metrics(v, mask, T)
        choices.append((I / L, mask))
    return max(choices, key=lambda row: row[0])[1]

## Prepare each trace

Normalize importance and convert the fitted KiB costs to row costs once per trace.

In [ ]:
cases = []
for trace in traces:
    v = trace["values"]
    v = (v / v.mean()).astype(np.float32).astype(float) if v.any() else v
    w = weight_bytes * trace["d"] / 1024
    s, b1, b2 = s_kib / w, w * c1, w * (c1 + delta)
    T = costs(len(v), s, b1, b2)
    cases.append((trace, v, w, s, b1, b2, T))

## Comparing ratios at the same budget

For a mask with exactly $R$ rows and ratio $u_M=I(M)/L(M)$, define $g(M)=100(1-u_M/\eta_R^\star)$. The oracle bracket gives

$$\max(0,100(1-u_M/\ell))\le g(M)\le\max(0,100(1-u_M/u)).$$

We report the upper end; the interval width is at most $100\epsilon$ percentage points in exact arithmetic. An underfilled Paper mask is outside the fixed-$R$ feasible set, so its gap is undefined. Empty selections and zero-importance cases also have no ratio gap, but retain their other measurements and status.

The CSV records the requested and selected counts, importance, latency and run intervals. Fragmentation is summarized by the number of runs and their total short-run deficit,

$$D(M)=\sum_{C\in\mathcal C(M)}(s-|C|)_+,\qquad
L(M)=b_2|M|+(b_2-b_1)D(M).$$

This identity also applies to underfilled masks. The analysis notebook reconstructs mask images from the saved intervals. The save cell writes the completed comparison to CSV.

In [ ]:
def mask_stats(v, mask, T, s, penalty, R, lower, upper):
    selected = int(mask.sum())
    I, L = metrics(v, mask, T)
    spans = runs(mask)
    lengths = np.diff(spans, axis=1).ravel()
    shortfall = np.maximum(s - lengths, 0).sum()
    comparable = selected == R and lower > 0
    status = "empty_selection" if selected == 0 else "ok" if v.any() else "zero_importance"
    return dict(
        R_selected=selected, fill_pct=100*selected/R, status=status,
        importance=I, latency=L, retained_pct=100*I/v.sum() if v.any() else np.nan,
        chunks=len(spans), short_chunks=int((lengths < s).sum()),
        shortfall_rows=shortfall, overhead_ms=penalty*shortfall,
        gap_lower_pct=max(0, 100*(1-I/L/lower)) if comparable else np.nan,
        gap_upper_pct=max(0, 100*(1-I/L/upper)) if comparable else np.nan,
        intervals=json.dumps(spans.tolist()))

## Run the comparison

In [ ]:
paper = load_paper()

In [ ]:
records = []
for trace, v, w, s, b1, b2, T in cases:
    for fraction in BUDGETS:
        R = max(1, int(fraction * len(v)))
        oracle, lower, upper, calls = solve(v, R, s, b1, b2)
        masks = {
            "Global DP": oracle,
            "Paper greedy": paper(v, R, w, trace["shape"]),
            "Saturation tiles": tiles(v, R, T, int(np.ceil(s))),
            "Top-R": top_r(v, R),
        }
        metadata = dict(trace_id=trace["trace_id"], shape=trace["shape"],
            prompt=trace["input_index"], fraction=fraction, N=len(v),
            R_nominal=R, oracle_calls=calls)
        for name, mask in masks.items():
            stats = mask_stats(v, mask, T, s, b2-b1, R, lower, upper)
            records.append(dict(**metadata, method=name, **stats))

## Inspect the results

In [ ]:
results = pd.DataFrame(records)
results.head()

## Save the CSV

This cell replaces `oracle_results_fixed_r.csv`. Continue with [02_analyze_oracle.ipynb](02_analyze_oracle.ipynb).

In [ ]:
results.to_csv(RESULTS, index=False)